# REHAB24-6 HRNet 2D skeleton features + paired LOSO vs RTMPose (Colab GPU)

Swaps the RTMPose backbone for **HRNet (whole-body)** and asks one question with the right control:

> at the *same* 2D-only feature pipeline, does a stronger top-down backbone (HRNet) recover more squat-correctness signal than RTMPose?

**Why this is the fair comparison.** The existing experiment already showed RTMPose-2D (LOSO 0.570) sits *below* MediaPipe pseudo-3D (0.633) — not because RTMPose is weak, but because the 2D-only path drops BlazePose's world-depth channel. HRNet stays 2D-only too, so the honest control is **HRNet-2D vs RTMPose-2D** (this notebook), which isolates *backbone accuracy* from the depth factor. Comparing HRNet-2D against MediaPipe would confound the two.

**Expected payoff is small and may be inside the noise.** Per-fold LOSO std is ~±0.05; a backbone swap is worth ~+0.01–0.03. So we report per-fold paired deltas + Wilcoxon, not a single number.

## Runtime differences from the RTMPose notebook
HRNet is **not** in `rtmlib`. It runs through the `mmpose` runtime (`MMPoseInferencer`), which needs the full OpenMMLab stack (mmengine + mmcv + mmdet + mmpose). That stack is heavier and more version-sensitive on Python 3.12 than rtmlib (the `src/pose/mmpose_pose_extraction.py` shims exist for exactly this). Use a **GPU runtime** (`Runtime > Change runtime type > GPU`).

## Prerequisites
- Repo uploaded to Drive (e.g. `MyDrive/x-coach/`) with `data/REHAB24-6/processed/manifest.csv` + `splits/` + `labels/correctness.json`.
- The **RTMPose features must already exist** (the rtmlib notebook's output): `data/REHAB24-6/processed/mmpose_skeleton_features/`. The paired comparison reads them as the baseline.

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Install the OpenMMLab stack (HRNet needs mmpose, not rtmlib).
#    mim resolves an mmcv wheel matching Colab's torch/CUDA. If a later cell fails to
#    `import mmpose`, the usual cause is an mmcv<->torch mismatch — see the note in cell 3.
!pip -q install -U openmim
!pip -q install -U setuptools==75.6.0
!pip -q install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121
!pip -q install numpy==1.26.4
!pip -q install mmengine
!TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2 pip -q install mmcv==2.1.0
!pip -q install mmdet==3.3.0
!pip -q install mmpose==1.3.2
!pip -q install numpy==1.26.4
!pip -q install "transformers==4.40.2" "tokenizers==0.19.1" "huggingface_hub==0.23.5"  # match torch 2.2.2: newer transformers needs torch>=2.3/2.4 (NameError nn / torch._C.Tag.needs_fixed_stride_order)
print('\n>>> Now RESTART the session: Runtime > Restart session.')
print('>>> Then run from cell 3 onward (skip cells 1-2; the Drive mount survives a restart).')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 21.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 42.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.5/314.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.4/239.4 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 9.3 MB/s eta 0:00:

In [3]:
# 3. Verify GPU + that mmpose imports cleanly. Run this AFTER the cell-2 restart.
#    The repo's import helper installs the Python 3.12 pkg_resources / xtcocotools shims.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU, then restart.'

import sys
from pathlib import Path
REPO_ON_DRIVE = Path('/content/drive/MyDrive/x-coach')   # <-- adjust if you uploaded elsewhere
assert REPO_ON_DRIVE.exists(), f'Repo not found on Drive: {REPO_ON_DRIVE}'
sys.path.insert(0, str(REPO_ON_DRIVE))

from src.pose.mmpose_pose_extraction import import_mmpose_inferencer
MMPoseInferencer = import_mmpose_inferencer()   # raises with a clear message if mmcv/mmpose mismatch
print('mmpose import OK — HRNet runtime ready.')
#  If this raises an mmcv error: `mim install "mmcv==2.1.0"` (pin), restart, re-run.

Tesla T4, 15360 MiB
torch 2.2.2+cu121 | cuda available: True


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# 4. Paths + copy the dataset to local disk (Drive FUSE is slow for video decode).
import shutil, time

LOCAL_DATA = Path('/content/REHAB24-6')
src_data = REPO_ON_DRIVE / 'data' / 'REHAB24-6'
assert (src_data / 'processed' / 'manifest.csv').exists(), 'manifest.csv missing — build it locally first.'
if not LOCAL_DATA.exists():
    print('Copying dataset to local disk (one-time, a few minutes)...')
    t = time.time(); shutil.copytree(src_data, LOCAL_DATA)
    print(f'  copied in {time.time()-t:.0f}s')
else:
    print('Local copy already present:', LOCAL_DATA)
print('videos:', len(list(LOCAL_DATA.rglob('*.mp4'))))

MANIFEST    = LOCAL_DATA / 'processed' / 'manifest.csv'
HRNET_DIR   = LOCAL_DATA / 'processed' / 'hrnet_skeleton_features'
RTMPOSE_DIR = LOCAL_DATA / 'processed' / 'mmpose_skeleton_features'

# Whole-body HRNet so the COCO-WholeBody->MediaPipe foot mapping (heel/toe) stays populated.
# Lighter alt: 'td-hm_hrnet-w32_dark-8xb64-210e_coco-wholebody-256x192'.
HRNET_MODEL = 'td-hm_hrnet-w48_dark-8xb32-210e_coco-wholebody-384x288'

In [ ]:
# 5. Smoke test: ONE video, end-to-end. Validates the mmpose runtime AND that
#    whole-body HRNet actually fills the foot landmarks (MediaPipe idx 29-32).
#    The first call downloads the HRNet config + weights and a default detector.
import numpy as np
from src.rehab24.mmpose_skeleton_features import extract_features_for_manifest
from src.rehab24.mediapipe_skeleton_features import MP_NUM_LANDMARKS
from src.rehab24.skeleton_features import SUMMARY_SIZE

written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=HRNET_DIR,
    runtime='mmpose', model=HRNET_MODEL, device='cuda:0', video_limit=1,
)
print('smoke-test reps written:', written)

sample = next(HRNET_DIR.rglob('*.npz'))
with np.load(sample) as d:
    f = d['video_feature']
print('sample', sample.name, '| feature_dim', f.shape[0], '| finite', bool(np.isfinite(f).all()))

# 33 joints x (x,y + vx,vy = 4 channels) x SUMMARY_SIZE(9) summary stats = 1188,
# identical to the RTMPose 2D-only branch (the classifier reads dim dynamically anyway).
EXPECTED_DIM = MP_NUM_LANDMARKS * 4 * SUMMARY_SIZE
print('expect feature_dim', EXPECTED_DIM, '(== RTMPose 2D-only).')
# Foot sanity check: a body-only (COCO-17) HRNet would leave heel/toe (MP 29-32)
# all-NaN -> interpolated to constants -> dead channels. Whole-body HRNet avoids this.
assert f.shape[0] == EXPECTED_DIM, f'unexpected dim {f.shape[0]} (expected {EXPECTED_DIM})'

In [ ]:
# 6. Full extraction (all 130 videos). Resumable: re-run after any disconnect; it
#    skips reps whose .npz already exists. HRNet-w48 top-down is slower than RTMPose
#    (per-person crop + heavy backbone) — expect several hours on a T4.
written = extract_features_for_manifest(
    data_root=LOCAL_DATA, manifest_path=MANIFEST, output_dir=HRNET_DIR,
    runtime='mmpose', model=HRNET_MODEL, device='cuda:0',
)
print('reps written this run:', written)
n = len(list(HRNET_DIR.rglob('*.npz')))
print('total .npz now:', n, '(expect 2144)')

In [ ]:
# 7. Verify all reps present and finite, then copy the (small) feature dir back to Drive.
paths = list(HRNET_DIR.rglob('*.npz'))
bad = [p.name for p in paths if not np.isfinite(np.load(p)['video_feature']).all()]
print(f'total={len(paths)}  non-finite={len(bad)}')
assert len(paths) == 2144 and not bad, f'CHECK: count={len(paths)} bad={bad[:5]}'

dst = REPO_ON_DRIVE / 'data' / 'REHAB24-6' / 'processed' / 'hrnet_skeleton_features'
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(HRNET_DIR, dst)
print('copied HRNet features to Drive:', dst)

## Paired LOSO: HRNet-2D vs RTMPose-2D

Runs Leave-One-Subject-Out **once per fold, training both feature sets on the identical train/val/test split and seed** so the per-fold deltas are properly paired (this cancels each subject's intrinsic difficulty, the dominant variance). We report:
- mean±std bal_acc per backbone (9 folds, excluding the under-powered P10, n=16),
- per-fold paired delta (HRNet − RTMPose) mean±std,
- Wilcoxon signed-rank over the folds (low power at n=9 — a non-significant result means "undetermined", not "no effect"),
- an **excluding-P5** read (P5 is a data ceiling: near-random for every feature, per the experiment summary).

The per-fold loop mirrors `src/rehab24/loso_cross_validation.py` (same primitives, same val-subject pick, same seed).

In [ ]:
# 8. Make sure the RTMPose baseline features are present locally (pull from Drive if needed).
if not RTMPOSE_DIR.exists():
    drive_rtm = REPO_ON_DRIVE / 'data' / 'REHAB24-6' / 'processed' / 'mmpose_skeleton_features'
    assert drive_rtm.exists(), ('RTMPose features not found. Run the rtmlib notebook '
                                '(rehab24_mmpose_colab.ipynb) first and copy them to Drive.')
    shutil.copytree(drive_rtm, RTMPOSE_DIR)
print('HRNet npz   :', len(list(HRNET_DIR.rglob('*.npz'))))
print('RTMPose npz :', len(list(RTMPOSE_DIR.rglob('*.npz'))))

In [ ]:
# 9. Paired LOSO driver.
import json
import numpy as np
import torch
from src.rehab24.loso_cross_validation import (
    FoldConfig, MIN_VAL_SUBJECT_SAMPLES, pick_val_subject, subjects_to_samples, train_one_fold,
)
from src.video.videomae_video_classifier import compute_metrics

LABELS = LOCAL_DATA / 'processed' / 'labels' / 'correctness.json'
FEATURE_DIRS = {'rtmpose': RTMPOSE_DIR, 'hrnet': HRNET_DIR}   # baseline first, candidate second
SEED = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = FoldConfig()   # same hyperparameters as the committed LOSO baselines

labels = {k: int(v) for k, v in json.load(LABELS.open()).items()}
subject_samples = subjects_to_samples(MANIFEST)
sample_counts = {p: len(ids) for p, ids in subject_samples.items()}
ordered = sorted(subject_samples, key=int)

folds = []
for test_subj in ordered:
    val_subj = pick_val_subject(test_subj, ordered, sample_counts)
    test_ids = subject_samples[test_subj]
    val_ids = subject_samples[val_subj]
    train_ids = [sid for s in ordered if s not in {test_subj, val_subj} for sid in subject_samples[s]]
    rec = {'test': test_subj, 'val': val_subj, 'n_test': len(test_ids)}
    for name, fdir in FEATURE_DIRS.items():
        thr, prob, lab = train_one_fold(fdir, train_ids, val_ids, test_ids, labels, config, device, SEED)
        rec[name] = compute_metrics(prob, lab, threshold=thr)['balanced_accuracy']
    rec['delta'] = rec['hrnet'] - rec['rtmpose']
    folds.append(rec)
    print(f"P{test_subj:<3} (val P{val_subj}, n={rec['n_test']:>4})  "
          f"rtmpose={rec['rtmpose']:.3f}  hrnet={rec['hrnet']:.3f}  Δ={rec['delta']:+.3f}")

In [ ]:
# 10. Summaries + Wilcoxon, saved to a JSON artifact.
def summarize(rows, key):
    a = np.asarray([r[key] for r in rows], float)
    return {'mean': float(a.mean()), 'std': float(a.std(ddof=0)), 'min': float(a.min()), 'max': float(a.max())}

def report(rows, title):
    rtm, hrn, dlt = summarize(rows, 'rtmpose'), summarize(rows, 'hrnet'), summarize(rows, 'delta')
    print(f"\n=== {title} ({len(rows)} folds) ===")
    print(f"  rtmpose bal_acc : {rtm['mean']:.3f} ± {rtm['std']:.3f}  (min {rtm['min']:.3f}, max {rtm['max']:.3f})")
    print(f"  hrnet   bal_acc : {hrn['mean']:.3f} ± {hrn['std']:.3f}  (min {hrn['min']:.3f}, max {hrn['max']:.3f})")
    print(f"  paired Δ (hrnet-rtmpose): {dlt['mean']:+.3f} ± {dlt['std']:.3f}  "
          f"({sum(r['delta'] > 0 for r in rows)}/{len(rows)} folds positive)")
    deltas = [r['delta'] for r in rows]
    try:
        from scipy.stats import wilcoxon
        if any(d != 0 for d in deltas):
            stat, p = wilcoxon(deltas)
            print(f"  Wilcoxon signed-rank: stat={stat:.1f}  p={p:.3f}  "
                  f"({'significant' if p < 0.05 else 'UNDETERMINED at n=' + str(len(rows))})")
        else:
            print('  Wilcoxon: all deltas zero — no difference.')
    except ImportError:
        print('  (scipy not available; install for Wilcoxon)')
    return {'rtmpose': rtm, 'hrnet': hrn, 'delta': dlt}

big = [r for r in folds if r['n_test'] >= MIN_VAL_SUBJECT_SAMPLES]          # drop tiny P10
no_p5 = [r for r in big if r['test'] != '5']                                # drop the data-ceiling subject
summary = {
    'all_10_folds': report(folds, 'all 10 folds'),
    'no_p10_9_folds': report(big, '9 folds (no P10)'),
    'no_p10_no_p5': report(no_p5, '9 folds minus P5 (data-ceiling subject)'),
}

out = REPO_ON_DRIVE / 'data' / 'REHAB24-6' / 'processed' / 'correctness_loso_hrnet_vs_rtmpose.json'
out.write_text(json.dumps({'seed': SEED, 'config': vars(config), 'folds': folds, 'summary': summary},
                          indent=2, sort_keys=True), encoding='utf-8')
print('\nsaved:', out)

## How to read this, and what's next

- **If the paired Δ is small and Wilcoxon is non-significant** (the likely outcome): a stronger 2D backbone alone does not move squat correctness — consistent with the experiment summary's conclusion that the bottleneck is the missing depth channel, not 2D keypoint accuracy. Report it as "undetermined / within noise", not "HRNet is worse".
- **Always also look per-exercise.** Squat (Ex6) is the format where 2D-only is strongest; upper-body actions (arm abduction/VW, push-ups) collapse without depth. A small overall Δ can hide a real per-action effect. Run the per-exercise breakdown locally after pulling the feature dir:
  ```bash
  # feature dir lands at data/REHAB24-6/processed/hrnet_skeleton_features/
  python scripts/rehab24/loso_per_exercise.py   # edit it to add the hrnet feature dir
  ```
- **The standalone single-dir LOSO** (matches the other committed baselines):
  ```bash
  python scripts/rehab24/loso_cross_validation.py \
    --feature-dir data/REHAB24-6/processed/hrnet_skeleton_features \
    --summary-output data/REHAB24-6/processed/correctness_loso_hrnet.json
  ```
- **If you want to actually beat MediaPipe** (not just RTMPose): HRNet's clean 2D is the right *input* to 2D→3D lifting (brief R5) — that's where the depth signal comes back. This notebook produces exactly that strong-2D source.